# Zain Jordan Customer 360 AI Workshop  
## Class 2: Building AI Agents with LangChain Tools

### Class Goal

In Class 1, we connected Python to the Zain Jordan Customer 360 SQLite database and created database functions.

In Class 2, we will turn those Python functions into **LangChain tools** and build a simple **Customer Care AI Agent**.

---

## What We Will Build

A beginner-friendly AI agent that can answer questions such as:

> Analyze customer 42. Check their profile, plan, churn risk, complaints, support history, and billing summary. Recommend the next best action.

---

## Important Boundary for Class 2

This class focuses only on:

- Python functions
- LangChain tools
- One AI agent
- Zain Jordan customer-care use case

We are **not** doing MCP, RAG, SQL Agent, or multi-agent in this class. Those come later.


# 1. Learning Outcomes

By the end of this notebook, you should be able to:

1. Understand what an AI agent is.
2. Understand what a LangChain tool is.
3. Convert Python database functions into LangChain tools.
4. Create a tool-using AI agent.
5. Ask the agent customer-care questions.
6. Use the Zain Jordan database as the source of truth.


# 2. Install Required Packages

Run this cell first in Google Colab.

If you already installed these packages, you can still run the cell. It will update/install what is needed.


In [ ]:
%pip install -q -U langchain langchain-openai langchain-community pandas


# 3. Import Libraries


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")


# 4. Set OpenAI API Key

In Google Colab:

1. Click the key icon on the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Paste your OpenAI API key.
4. Enable notebook access for the secret.

This notebook will read the API key from Colab Secrets.


In [ ]:
# Load OpenAI API key from Google Colab Secrets if available.
# If running locally, set OPENAI_API_KEY in your environment.

try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. The agent cells will not run until you set it.")
else:
    print("OPENAI_API_KEY is available.")


# 5. Upload or Locate the Zain Jordan Database

Upload the same database used in Class 1:

`zain_customer_360_ai_demo.db`

If your file has a different name, the notebook will try to detect any `.db` file in the current folder.


In [ ]:
# Run this cell if you are using Google Colab and need to upload the database file.

try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


# 6. Connect to the SQLite Database


In [ ]:
# Automatically detect a SQLite database file in the current folder.
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

print("Database path:", DB_PATH)
print("File exists:", Path(DB_PATH).exists())

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
print("Connected to Zain Jordan Customer 360 database.")


# 7. Quick Database Check

Let us confirm that the database is available and contains tables.


In [ ]:
tables_df = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
""", conn)

print("Number of tables:", len(tables_df))
tables_df


# 8. Helper Function: Convert DataFrame to Text

LangChain tools should return simple text or structured data.

For beginner training, we will return clear text that the AI model can read.


In [ ]:
def df_to_text(df, max_rows=10):
    """Convert a pandas DataFrame into readable text for an AI tool."""
    if df is None or df.empty:
        return "No records found."

    display_df = df.head(max_rows)
    return display_df.to_string(index=False)


# 9. Create Normal Python Database Functions First

Before turning functions into tools, we create normal Python functions.

This helps participants understand:

> A tool is just a function that the AI agent is allowed to call.


## Function 1: Customer Profile


In [ ]:
def fetch_customer_profile(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        c.gender,
        c.age_group,
        c.city,
        c.governorate,
        c.customer_type,
        c.customer_segment,
        c.preferred_language,
        c.status AS customer_status,
        a.account_type,
        a.account_status,
        a.credit_limit_jod
    FROM customers c
    LEFT JOIN accounts a
        ON c.customer_id = a.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


In [ ]:
print(fetch_customer_profile(42))


## Function 2: Customer Plan


In [ ]:
def fetch_customer_plan(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.msisdn,
        s.service_type,
        s.status AS subscription_status,
        p.plan_name,
        p.plan_category,
        p.monthly_fee_jod,
        p.data_allowance_gb,
        p.local_minutes,
        p.international_minutes,
        p.roaming_minutes,
        p.sms_allowance,
        p.technology,
        p.contract_months
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN plans p
        ON s.plan_id = p.plan_id
    WHERE c.customer_id = ?
    ORDER BY s.primary_subscription_flag DESC, s.activation_date DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


In [ ]:
print(fetch_customer_plan(42))


## Function 3: Customer Churn Risk


In [ ]:
def fetch_customer_churn_risk(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        ch.score_month,
        ch.churn_score,
        ch.risk_level,
        ch.main_risk_reason,
        ch.recommended_action
    FROM customer_churn_scores ch
    JOIN customers c
        ON ch.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


In [ ]:
print(fetch_customer_churn_risk(42))


## Function 4: Customer Complaints


In [ ]:
def fetch_customer_complaints(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        complaint_date,
        complaint_category,
        complaint_description,
        severity,
        status,
        resolved_date,
        compensation_amount_jod
    FROM complaints
    WHERE customer_id = ?
    ORDER BY complaint_date DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


In [ ]:
print(fetch_customer_complaints(42))


## Function 5: Customer Support History


In [ ]:
def fetch_customer_support_interactions(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        interaction_datetime,
        channel,
        reason_category,
        issue_type,
        priority,
        resolution_status,
        resolution_time_minutes,
        customer_sentiment
    FROM support_interactions
    WHERE customer_id = ?
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


In [ ]:
print(fetch_customer_support_interactions(42))


## Function 6: Customer Billing Summary


In [ ]:
def fetch_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        i.invoice_id,
        i.billing_period_start,
        i.billing_period_end,
        i.issue_date,
        i.due_date,
        i.total_amount_jod,
        i.payment_status,
        i.days_overdue
    FROM customers c
    JOIN accounts a
        ON c.customer_id = a.customer_id
    JOIN invoices i
        ON a.account_id = i.account_id
    WHERE c.customer_id = ?
    ORDER BY i.issue_date DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


In [ ]:
print(fetch_customer_billing_summary(42))


## Function 7: Customer Usage Summary


In [ ]:
def fetch_customer_usage_summary(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.service_type,
        COUNT(d.session_id) AS total_data_sessions,
        ROUND(SUM(d.data_used_mb) / 1024.0, 2) AS total_data_used_gb,
        ROUND(SUM(d.cost_jod), 2) AS total_data_cost_jod,
        MAX(d.session_start_time) AS last_data_session
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    LEFT JOIN data_usage_sessions d
        ON s.subscription_id = d.subscription_id
    WHERE c.customer_id = ?
    GROUP BY c.customer_id, c.full_name, s.subscription_id, s.service_type
    ORDER BY total_data_used_gb DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


In [ ]:
print(fetch_customer_usage_summary(42))


# 10. Convert Python Functions into LangChain Tools

Now we convert the normal functions into LangChain tools.

The tool description is very important.

The agent reads the tool name and description to decide when to use the tool.


In [ ]:
from langchain.tools import tool

@tool
def get_customer_profile(customer_id: int) -> str:
    """Get customer profile, city, segment, language, account type, and account status for a Zain Jordan customer ID."""
    return fetch_customer_profile(customer_id)


@tool
def get_customer_plan(customer_id: int) -> str:
    """Get active subscriptions, mobile numbers, service type, current plans, monthly fee, data allowance, minutes, and contract details for a customer ID."""
    return fetch_customer_plan(customer_id)


@tool
def get_customer_churn_risk(customer_id: int) -> str:
    """Get churn score, churn risk level, main risk reason, and recommended retention action for a customer ID."""
    return fetch_customer_churn_risk(customer_id)


@tool
def get_customer_complaints(customer_id: int, limit: int = 5) -> str:
    """Get recent customer complaints including complaint category, description, severity, status, resolved date, and compensation amount."""
    return fetch_customer_complaints(customer_id, limit)


@tool
def get_customer_support_interactions(customer_id: int, limit: int = 5) -> str:
    """Get recent support interactions including channel, issue type, priority, resolution status, resolution time, and customer sentiment."""
    return fetch_customer_support_interactions(customer_id, limit)


@tool
def get_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    """Get recent invoice and billing summary including invoice dates, total amount, payment status, and days overdue for a customer ID."""
    return fetch_customer_billing_summary(customer_id, limit)


@tool
def get_customer_usage_summary(customer_id: int) -> str:
    """Get customer data usage summary including data sessions, total data used in GB, data cost, and last data session for a customer ID."""
    return fetch_customer_usage_summary(customer_id)


# 11. Inspect the Tools

This helps beginners see that each tool has a name and description.


In [ ]:
tools = [
    get_customer_profile,
    get_customer_plan,
    get_customer_churn_risk,
    get_customer_complaints,
    get_customer_support_interactions,
    get_customer_billing_summary,
    get_customer_usage_summary,
]

for t in tools:
    print("Tool name:", t.name)
    print("Description:", t.description)
    print("-" * 80)


# 12. Create the Language Model and First Agent

We will use LangChain's `create_agent`.

You can change the model name below based on your available OpenAI model.

Examples:

- `openai:gpt-4.1-mini`
- `openai:gpt-4o-mini`
- `openai:gpt-5.4` if available in your account


In [ ]:
from langchain.agents import create_agent

MODEL_NAME = "openai:gpt-4.1-mini"

basic_system_prompt = """
You are a professional telecom customer-care AI assistant for Zain Jordan.

You must use the available tools when customer data is needed.
Do not guess customer information.
If a tool returns no records, clearly say that data was not found.
Keep answers clear, structured, and business-friendly.
"""

basic_agent = create_agent(
    model=MODEL_NAME,
    tools=[
        get_customer_profile,
        get_customer_churn_risk,
    ],
    system_prompt=basic_system_prompt,
)

print("Basic agent created successfully.")


# 13. Helper Function to Run the Agent

This helper extracts the final answer from the agent response.


In [ ]:
def extract_final_text(result):
    """Extract readable final text from a LangChain agent result."""
    last_message = result["messages"][-1]

    if hasattr(last_message, "content") and isinstance(last_message.content, str):
        return last_message.content

    if hasattr(last_message, "content_blocks"):
        parts = []
        for block in last_message.content_blocks:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))
            else:
                parts.append(str(block))
        return "\\n".join(parts)

    return str(last_message)


def run_agent(agent, question):
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return extract_final_text(result)


# 14. Demo 1: Simple Tool-Using Agent

The agent has only two tools:

- customer profile
- churn risk

Ask a simple question.


In [ ]:
question = "Who is customer 42 and what is their churn risk?"

answer = run_agent(basic_agent, question)
print(answer)


# 15. Create the Full Customer Care Agent

Now we give the agent more tools.

This is the main Class 2 demo.


In [ ]:
customer_care_system_prompt = """
You are a professional telecom customer-care AI assistant for Zain Jordan.

Your job is to help customer-care teams understand customers and recommend the next best action.

Rules:
1. Use tools whenever customer data is needed.
2. Do not invent customer facts.
3. If a tool returns no data, say the data was not available.
4. For customer analysis, check profile, plan, churn risk, complaints, support history, billing, and usage when relevant.
5. Provide a clear final answer with these sections:
   - Customer Summary
   - Plan Summary
   - Risk Summary
   - Customer Experience Summary
   - Billing Summary
   - Usage Summary
   - Recommended Next Action
   - Suggested Customer-Care Message
6. Keep the tone professional, concise, and business-friendly.
"""

customer_care_agent = create_agent(
    model=MODEL_NAME,
    tools=tools,
    system_prompt=customer_care_system_prompt,
)

print("Full Customer Care AI Agent created successfully.")


# 16. Demo 2: Full Customer Analysis

This is the main demo prompt for Class 2.


In [ ]:
question = """
Analyze customer 42.

Check their profile, current plan, churn risk, recent complaints, support history, billing summary, and usage summary.

Then recommend the next best action and draft a short professional customer-care message.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


# 17. Demo 3: Compare Two Customers

This shows the agent can use tools repeatedly for different customers.


In [ ]:
question = """
Compare customer 25 and customer 42.

For each customer, check churn risk and recent complaints.
Tell me which customer should be prioritized first by the retention team and why.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


# 18. Demo 4: Billing-Focused Question

This shows the agent can answer a specific customer-care question.


In [ ]:
question = """
Customer 42 is asking about billing.

Check their recent invoices and payment status.
Explain whether there is any billing concern and suggest what the support agent should say.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


# 19. Demo 5: Usage-Focused Question

This shows the agent can use the usage summary tool.


In [ ]:
question = """
Customer 42 wants to know whether their current plan fits their data usage.

Check their plan and usage summary.
Suggest whether the customer may need a better plan, add-on, or no change.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


# 20. Optional: See Full Agent Messages

Sometimes trainers may want to see the tool calls and intermediate messages.

This cell prints the message objects returned by the agent.


In [ ]:
debug_result = customer_care_agent.invoke({
    "messages": [
        {"role": "user", "content": "Check the churn risk for customer 42."}
    ]
})

for i, message in enumerate(debug_result["messages"]):
    print(f"--- Message {i+1} ---")
    print(message)
    print()


# 21. Exercise 1: Manual Tool Selection

Before running the agent, decide which tool should be used.

| User Question | Correct Tool |
|---|---|
| Who is customer 42? | `get_customer_profile` |
| What plan does customer 42 have? | `get_customer_plan` |
| Is customer 42 likely to leave? | `get_customer_churn_risk` |
| Did customer 42 complain recently? | `get_customer_complaints` |
| Did customer 42 contact support? | `get_customer_support_interactions` |
| Does customer 42 have unpaid bills? | `get_customer_billing_summary` |
| How much data did customer 42 use? | `get_customer_usage_summary` |


# 22. Exercise 2: Analyze Different Customers

Try these customer IDs:

- 10
- 25
- 42
- 100

Ask the agent to analyze each customer and identify who needs urgent action.


In [ ]:
question = """
Analyze customer 10.

Check profile, plan, churn risk, complaints, support history, billing, and usage.
Recommend the next best action.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


In [ ]:
question = """
Analyze customer 25.

Check profile, plan, churn risk, complaints, support history, billing, and usage.
Recommend the next best action.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


In [ ]:
question = """
Analyze customer 100.

Check profile, plan, churn risk, complaints, support history, billing, and usage.
Recommend the next best action.
"""

answer = run_agent(customer_care_agent, question)
print(answer)


# 23. Exercise 3: Improve the System Prompt

Try changing the system prompt.

Example:

> You are a senior telecom retention specialist.  
> Focus only on churn risk, complaints, and retention recommendation.  
> Keep the answer short and action-oriented.

Then recreate the agent and test again.


In [ ]:
retention_system_prompt = """
You are a senior telecom retention specialist for Zain Jordan.

Focus on:
1. Churn risk
2. Complaint history
3. Support history
4. Recommended retention action

Do not guess.
Use tools for customer facts.
Keep the answer short, direct, and action-oriented.
"""

retention_agent = create_agent(
    model=MODEL_NAME,
    tools=[
        get_customer_profile,
        get_customer_churn_risk,
        get_customer_complaints,
        get_customer_support_interactions,
    ],
    system_prompt=retention_system_prompt,
)

question = """
Customer 42 may be at risk.
Check the customer profile, churn risk, complaints, and support history.
Recommend a retention action.
"""

answer = run_agent(retention_agent, question)
print(answer)


# 24. Exercise 4: Create a New Tool

In this exercise, participants can create a new tool.

Suggested tool:

`get_customer_value_segment`

Purpose:

- Get ARPU
- Get six-month revenue
- Get value segment
- Help prioritize high-value customers


In [ ]:
def fetch_customer_value_segment(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        v.segment_month,
        v.arpu_jod,
        v.total_revenue_6m_jod,
        v.value_segment,
        v.lifetime_months
    FROM customer_value_segments v
    JOIN customers c
        ON v.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


@tool
def get_customer_value_segment(customer_id: int) -> str:
    """Get customer value segment, ARPU, six-month revenue, and lifetime months for a Zain Jordan customer ID."""
    return fetch_customer_value_segment(customer_id)


print(fetch_customer_value_segment(42))


# 25. Optional: Build a Value-Aware Retention Agent

This agent can prioritize customers using both churn risk and value segment.


In [ ]:
value_aware_retention_agent = create_agent(
    model=MODEL_NAME,
    tools=[
        get_customer_profile,
        get_customer_churn_risk,
        get_customer_complaints,
        get_customer_support_interactions,
        get_customer_value_segment,
    ],
    system_prompt="""
You are a Zain Jordan retention prioritization assistant.

Use tools to check:
- customer profile
- churn risk
- complaints
- support history
- customer value segment

Prioritize customers who are both high churn risk and high value.
Always explain why the customer should or should not be prioritized.
""",
)

question = """
Compare customer 25, customer 42, and customer 100.
Which one should the retention team prioritize first?
"""

answer = run_agent(value_aware_retention_agent, question)
print(answer)


# 26. Common Errors and Fixes

## Error: OPENAI_API_KEY not found

Fix:

- Add `OPENAI_API_KEY` in Colab Secrets
- Enable notebook access
- Rerun the API key cell

## Error: database file not found

Fix:

- Upload the `.db` file again
- Check `DB_PATH`
- Confirm `Path(DB_PATH).exists()` returns `True`

## Error: tool gives no records

Fix:

- Try another customer ID
- Check that the customer exists
- Some customers may not have complaints or support interactions

## Error: model not available

Fix:

- Change `MODEL_NAME`
- Try `openai:gpt-4.1-mini` or another model available in your account


# 27. What We Built Today

In Class 2, we built:

1. Python database functions
2. LangChain tools
3. A basic tool-using agent
4. A full Customer Care AI Agent
5. A retention-focused agent
6. A value-aware retention agent

This is the foundation for the next classes.

---

## Next Class

In Class 3, we will build a **Natural Language SQL Agent**.

Instead of only using predefined tools, the SQL Agent will be able to inspect the database and generate SQL queries to answer broader business questions.


# 28. Trainer Closing Script

Today, we moved from Python functions to AI tools.

The important idea is simple:

> A tool is a function the AI agent can call.

Once we gave the agent tools, it could check customer profile, plan, churn risk, complaints, support history, billing, and usage.

This is the foundation for more advanced systems.

Next, we will move from controlled tools to a natural language SQL agent. Later, we will build RAG, multi-agent workflows, and MCP.
